# Track A: Deep LSTM + GNN 空间模块
# 特征工程 v2 — Tier A (~28维) / Tier B (~42维)
# 
# 运行环境: Colab (T4 GPU) 或本地 (MPS/CPU)
# 预计运行时间: ~15 min (Phase 1 LSTM) + ~10 min (Phase 2 GNN)

---
## §0 Config

In [ ]:
# ============================================================
# §0 Config — 只改这里! 其余 cell 全自动
# ============================================================

MODEL_NAME = "DeepLSTM"
EXPERIMENT_TAG = "v2"                # v2 特征工程

MODEL_TYPE = "lstm"
GNN_ENABLED = False                  # Phase 1 先不用, Phase 2 (§P2) 自动开启
TWO_STAGE_ENABLED = False
ENSEMBLE_ENABLED = False

# 数据
DATA_DIR = "data"
RESULTS_DIR = "results"
RANDOM_SEED = 42
VALIDATION_SPLIT = 0.2
OUTAGE_LAGS = (1, 3, 6, 12, 24)
ROLLING_WINDOWS = (6, 12, 24)

# 序列
SEQ_LEN = 48
HORIZON = 48
TRAIN_STRIDE = 6                     # 训练集每6h取一个样本 (去掉95%冗余!)
                                     # 验证集仍 stride=1 保证完整评估

# 训练
BATCH_SIZE = 128
EPOCHS = 200                         # 去冗余后一个epoch数据量少了, 需要更多epoch
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-3
LR_SCHEDULER = "cosine"
EARLY_STOPPING_PATIENCE = 30        # 数据少了收敛慢, 多给耐心
GRADIENT_CLIP = 1.0
INPUT_NOISE_STD = 0.05               # 训练时给输入加高斯噪声 (数据增强)

# ---- Deep LSTM 架构 ----
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.4
USE_GRU = False
BIDIRECTIONAL = False

# ---- GNN (Phase 2 自动启用) ----
GNN_HIDDEN = 64
GNN_K = 5
SPATIAL_FUSION = "gate"

# Transformer / Seq2Seq (本 notebook 不用, 占位)
N_HEADS = 8
DIM_FEEDFORWARD = 512
USE_CAUSAL_MASK = True
AUTOREGRESSIVE = False
TEACHER_FORCING = 0.5
CLS_THRESHOLD = 0.5
CLS_POS_WEIGHT = 3.0
ENSEMBLE_METHOD = "stacking"

# ---- 特征层级 (v2) ----
FEATURE_TIER = "B"                   # "A" = compact ~28 维 (~94%)
                                     # "B" = full ~42 维 (~99%)

# 目标变换
LOG_TRANSFORM_Y = True               # log1p(out), 推理时自动 expm1 还原

# 损失函数
LOSS_FN = "huber"
HUBER_DELTA = 5.0                    # log 空间

EXTREME_WEIGHT = 5.0
EXTREME_THRESHOLD = 500

# 设备
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"=== {MODEL_NAME} ({EXPERIMENT_TAG}) ===")
print(f"Device: {DEVICE} | LSTM: {NUM_LAYERS}层×{HIDDEN_DIM}dim | Dropout: {DROPOUT}")
print(f"Feature Tier: {FEATURE_TIER} | SEQ_LEN: {SEQ_LEN} | HORIZON: {HORIZON}")
print(f"LR: {LEARNING_RATE} | WD: {WEIGHT_DECAY} | Batch: {BATCH_SIZE}")
print(f"TRAIN_STRIDE: {TRAIN_STRIDE} | INPUT_NOISE: {INPUT_NOISE_STD}")
print(f"Epochs: {EPOCHS} | Early Stop: {EARLY_STOPPING_PATIENCE}")

---
## §1 数据加载 + 特征工程 (自动，不用改)

In [ ]:
# §1.1 导入 + 环境配置

# ---- Colab 自动检测 ----
import os, sys
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    # Colab: 安装依赖 + 挂载 Drive
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'netCDF4', 'xarray', 'tqdm'], check=True)
    
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # 数据路径: 把 train.nc 放在 Google Drive 的 MLPS_Final_Project/data/ 下
    DRIVE_PROJECT = '/content/drive/MyDrive/MLPS_Final_Project'
    if os.path.exists(os.path.join(DRIVE_PROJECT, 'data', 'train.nc')):
        DATA_DIR = os.path.join(DRIVE_PROJECT, 'data')
        RESULTS_DIR = os.path.join(DRIVE_PROJECT, 'results')
        print(f"✓ 使用 Drive 数据: {DATA_DIR}")
    elif os.path.exists('data/train.nc'):
        print("✓ 使用本地 data/")
    else:
        print("⚠ 找不到 train.nc! 请上传到 data/ 或 Drive/MLPS_Final_Project/data/")
    
    print(f"✓ Colab 环境就绪")
else:
    print(f"✓ 本地环境")

import time, warnings, platform
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed_all(RANDOM_SEED)

# 中文字体
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Arial Unicode MS']
elif platform.system() == 'Linux':
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False

print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"  Memory: {mem / 1e9:.1f} GB")

In [ ]:
# §1.2 加载数据
ds = xr.open_dataset(os.path.join(DATA_DIR, 'train.nc'))
timestamps = pd.to_datetime(ds.timestamp.values)
locations = list(ds.location.values)
weather_features = list(ds.feature.values)
outage  = ds.out.transpose('timestamp','location').values.astype(float)
tracked = ds.tracked.transpose('timestamp','location').values.astype(float)
weather = ds.weather.transpose('timestamp','location','feature').values.astype(float)
T, L = outage.shape
print(f"T={T}h | L={L} counties | F={len(weather_features)} weather features")
print(f"时间范围: {timestamps[0]} ~ {timestamps[-1]}")

In [ ]:
# §1.3 Phase 1 关键产出 (GMM 分层 + 天气特征筛选)
from sklearn.mixture import GaussianMixture
from scipy.stats import spearmanr

# GMM regime thresholds
nonzero_out = outage[outage > 0].flatten()
log_nz = np.log1p(nonzero_out).reshape(-1, 1)
best_k, best_bic = 2, np.inf
for k in range(2, 6):
    gm = GaussianMixture(n_components=k, random_state=RANDOM_SEED, max_iter=300).fit(log_nz)
    bic = gm.bic(log_nz)
    if bic < best_bic: best_k, best_bic = k, bic
gm_final = GaussianMixture(n_components=best_k, random_state=RANDOM_SEED, max_iter=300).fit(log_nz)
sorted_means = np.sort(gm_final.means_.flatten())
gmm_thresholds = [int(np.expm1((sorted_means[i]+sorted_means[i+1])/2)) for i in range(len(sorted_means)-1)]
print(f"GMM k={best_k}, thresholds={gmm_thresholds}")

# Weather feature selection (Spearman + collinearity dedup)
flat_out = outage.flatten()
sample_n = min(200_000, len(flat_out))
sidx = np.random.choice(len(flat_out), sample_n, replace=False)
corr_dict = {}
for fi, fn in enumerate(weather_features):
    rho, _ = spearmanr(flat_out[sidx], weather[:,:,fi].flatten()[sidx])
    corr_dict[fn] = abs(rho)
corr_series = pd.Series(corr_dict).sort_values(ascending=False)

selected_features = []
for feat in corr_series.index:
    fi = weather_features.index(feat)
    if not any(abs(np.corrcoef(
        weather[:,:,fi].flatten()[sidx],
        weather[:,:,weather_features.index(s)].flatten()[sidx])[0,1]) > 0.85
        for s in selected_features):
        selected_features.append(feat)
top_weather_features = list(corr_series.head(6).index)
print(f"去冗余后: {len(selected_features)} features | Top-6: {top_weather_features}")

In [ ]:
# §1.4 build_features() v2 — 精简版 (PCA + Tier A/B)
from sklearn.decomposition import PCA

def build_features(ds, sel_weather, top_weather, thresholds,
                   o_lags=(1,3,6,12,24), r_wins=(6,12,24), n_pca=3):
    """v2 特征构造: 193维→Tier A(~30)/Tier B(~42), 含天气PCA压缩"""
    ts = pd.to_datetime(ds.timestamp.values)
    locs = list(ds.location.values)
    fnames = list(ds.feature.values)
    out = ds.out.transpose('timestamp','location').values.astype(float)
    trk = ds.tracked.transpose('timestamp','location').values.astype(float)
    w   = ds.weather.transpose('timestamp','location','feature').values.astype(float)
    Tl, Ll = out.shape
    sel_weather = sel_weather or fnames
    top_weather = top_weather or fnames[:6]
    top_in = [f for f in top_weather[:6] if f in fnames]
    thresholds = thresholds or []
    
    # PCA on weather
    sel_idx = [fnames.index(f) for f in sel_weather if f in fnames]
    w_sel = w[:,:,sel_idx].reshape(Tl*Ll, len(sel_idx))
    w_sel = np.nan_to_num(w_sel, nan=0.0)
    w_mu, w_sig = w_sel.mean(0), w_sel.std(0); w_sig[w_sig==0]=1.
    w_norm = (w_sel - w_mu) / w_sig
    n_comp = min(n_pca, len(sel_idx))
    pca = PCA(n_components=n_comp, random_state=42)
    w_pca = pca.fit_transform(w_norm).reshape(Tl, Ll, n_comp)
    pca_info = {'pca_model':pca, 'w_mean':w_mu, 'w_std':w_sig,
                'n_components':n_comp, 'explained_variance_ratio':pca.explained_variance_ratio_,
                'cumulative_variance':np.cumsum(pca.explained_variance_ratio_)}
    print(f'  PCA: {len(sel_idx)}→{n_comp} dims, cumvar={pca_info["cumulative_variance"][-1]:.1%}')
    
    records = []
    for li in range(Ll):
        loc = locs[li]
        d = pd.DataFrame({'timestamp':ts,'location':loc,'out':out[:,li],'tracked':trk[:,li]})
        d['hour_sin']=np.sin(2*np.pi*ts.hour/24); d['hour_cos']=np.cos(2*np.pi*ts.hour/24)
        d['dow_sin']=np.sin(2*np.pi*ts.dayofweek/7); d['dow_cos']=np.cos(2*np.pi*ts.dayofweek/7)
        o_s = d['out']; rate = d['out']/d['tracked'].replace(0,np.nan)
        for lag in o_lags:
            d[f'out_lag_{lag}h']=o_s.shift(lag); d[f'rate_lag_{lag}h']=rate.shift(lag)
        so = o_s.shift(1)
        for win in r_wins:
            roll=so.rolling(win,min_periods=1)
            d[f'out_rmean_{win}h']=roll.mean(); d[f'out_rmax_{win}h']=roll.max()
            d[f'out_rstd_{win}h']=roll.std().fillna(0); d[f'out_rsum_{win}h']=roll.sum()
        for bw in [6,12,24]:
            d[f'had_outage_{bw}h']=(so.rolling(bw,min_periods=1).max()>0).astype(int)
        if thresholds:
            lo=o_s.shift(1); reg=pd.Series(np.zeros(Tl),index=d.index)
            for th in thresholds: reg=reg+(lo>th).astype(int)
            d['out_regime']=reg; d['out_regime_max_24h']=d['out_regime'].rolling(24,min_periods=1).max()
        for pc in range(n_comp): d[f'weather_pc{pc+1}']=w_pca[:,li,pc]
        for fn in top_in:
            fi=fnames.index(fn); ws=pd.Series(w[:,li,fi])
            r24=ws.rolling(24,min_periods=1)
            d[f'w_{fn}_rmean_24h']=r24.mean(); d[f'w_{fn}_rmax_24h']=r24.max()
        if len(top_in)>=3:
            zs=[]
            for fn in top_in[:5]:
                fi=fnames.index(fn); wc=w[:,li,fi]; wm=np.nanmean(wc); ws_=np.nanstd(wc)
                zs.append((wc-wm)/ws_ if ws_>0 else np.zeros(Tl))
            zm=np.array(zs)
            d['n_weather_anomaly']=(np.abs(zm)>1.5).sum(0)
            d['weather_anomaly_score']=np.abs(zm).mean(0)
            d['anomaly_score_max_6h']=d['weather_anomaly_score'].rolling(6,min_periods=1).max()
        records.append(d)
    
    df = pd.concat(records, ignore_index=True)
    
    lag_cols = [f'out_lag_{l}h' for l in o_lags]+[f'rate_lag_{l}h' for l in o_lags]
    roll_cols = []
    for win in r_wins: roll_cols += [f'out_rmean_{win}h',f'out_rmax_{win}h',f'out_rstd_{win}h',f'out_rsum_{win}h']
    roll_cols += [f'had_outage_{bw}h' for bw in [6,12,24]]
    storm_cols = ['n_weather_anomaly','weather_anomaly_score','anomaly_score_max_6h']
    tier_compact = [c for c in lag_cols+roll_cols+storm_cols if c in df.columns]
    
    regime_cols = ['out_regime','out_regime_max_24h'] if thresholds else []
    time_cols = ['hour_sin','hour_cos','dow_sin','dow_cos']
    wroll_cols = [c for c in df.columns if c.startswith('w_') and ('rmean_24h' in c or 'rmax_24h' in c)]
    pca_cols = [f'weather_pc{pc+1}' for pc in range(n_comp)]
    tier_full = [c for c in tier_compact+regime_cols+time_cols+wroll_cols+pca_cols if c in df.columns]
    
    print(f'  Tier A: {len(tier_compact)} dims | Tier B: {len(tier_full)} dims')
    return df, tier_compact, tier_full, pca_info

print('构造特征中...')
t0=time.time()
df, tier_compact, tier_full, pca_info = build_features(
    ds, selected_features, top_weather_features, gmm_thresholds,
    OUTAGE_LAGS, ROLLING_WINDOWS)
print(f'✓ {time.time()-t0:.0f}s | {df.shape[0]:,} rows')

In [ ]:
# §1.5 特征选择 + 训练/验证划分 + 标准化

# ---- 根据 FEATURE_TIER 选择特征列 (v2) ----
if FEATURE_TIER == "A":
    feature_cols = tier_compact
elif FEATURE_TIER == "B":
    feature_cols = tier_full
else:
    feature_cols = tier_full
    print(f'⚠ 未知 FEATURE_TIER="{FEATURE_TIER}", 使用 Tier B')

print(f'FEATURE_TIER = "{FEATURE_TIER}" → {len(feature_cols)} 维特征')

# ---- 训练/验证划分 ----
split_idx = int(T * (1 - VALIDATION_SPLIT))
split_time = timestamps[split_idx]
train_df = df[df['timestamp'] < split_time].dropna(subset=feature_cols).copy()
val_df = df[df['timestamp'] >= split_time].copy()
for c in feature_cols: val_df[c] = val_df[c].fillna(0)

scaler = StandardScaler().fit(train_df[feature_cols].values)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
print(f'y_train: mean={train_df["out"].mean():.1f}, nonzero={train_df["out"].gt(0).mean():.1%}')
print(f'y_val:   mean={val_df["out"].mean():.1f}, nonzero={val_df["out"].gt(0).mean():.1%}')

---
## §2 DataLoader

In [ ]:
# §2.1 DataFrame → 3D 张量
def df_to_3d(sub_df, locs, feat_cols, scaler_obj, log_y=False):
    ts_list = sorted(sub_df['timestamp'].unique())
    ts2i = {ts:i for i,ts in enumerate(ts_list)}
    loc2i = {l:i for i,l in enumerate(locs)}
    Ts, Ls, Fs = len(ts_list), len(locs), len(feat_cols)
    X = np.zeros((Ts,Ls,Fs), dtype=np.float32)
    Y = np.zeros((Ts,Ls), dtype=np.float32)
    sorted_df = sub_df.sort_values(['timestamp','location'])
    scaled = scaler_obj.transform(sorted_df[feat_cols].fillna(0).values)
    for idx, (_, row) in enumerate(sorted_df.iterrows()):
        ti, li = ts2i.get(row['timestamp']), loc2i.get(row['location'])
        if ti is not None and li is not None:
            X[ti,li,:] = scaled[idx]
            Y[ti,li] = np.log1p(row['out']) if log_y else row['out']
    return X, Y

X_train_3d, Y_train_3d = df_to_3d(train_df, locations, feature_cols, scaler, log_y=LOG_TRANSFORM_Y)
X_val_3d, Y_val_3d = df_to_3d(val_df, locations, feature_cols, scaler, log_y=LOG_TRANSFORM_Y)

# Y_val_3d_raw: 始终保留原始值, 用于最终 RMSE 评估
_, Y_val_3d_raw = df_to_3d(val_df, locations, feature_cols, scaler, log_y=False)

print(f'X_train: {X_train_3d.shape} | X_val: {X_val_3d.shape}')
if LOG_TRANSFORM_Y:
    print(f'  Y 已做 log1p 变换: range [{Y_train_3d.min():.2f}, {Y_train_3d.max():.2f}] (原始 max={np.expm1(Y_train_3d.max()):.0f})')
else:
    print(f'  Y 原始值: range [{Y_train_3d.min():.0f}, {Y_train_3d.max():.0f}]')

In [ ]:
# §2.2 PyTorch Dataset (支持 stride + 输入噪声)
class OutageDataset(Dataset):
    """
    mode='per_county':    每样本 = 单县时间窗 → (seq_len, F) → (horizon,)
    mode='all_counties':  每样本 = 全部县同一时间窗 → (seq_len, L, F) → (horizon, L)
    
    stride: 训练集用 stride>1 去掉冗余 (相邻样本 overlap 太多导致过拟合)
    noise_std: 训练时给输入加高斯噪声 (数据增强)
    """
    def __init__(self, X, Y, seq_len, horizon, mode='per_county', stride=1, noise_std=0.0):
        self.X, self.Y = X, Y
        self.seq_len, self.horizon, self.mode = seq_len, horizon, mode
        self.noise_std = noise_std
        self.Td, self.Ld, self.Fd = X.shape
        self.starts = list(range(0, self.Td - seq_len - horizon + 1, stride))
        self.n = len(self.starts) * (self.Ld if mode == 'per_county' else 1)

    def __len__(self): return self.n

    def __getitem__(self, idx):
        if self.mode == 'per_county':
            si = idx // self.Ld  # which start index
            li = idx % self.Ld   # which county
            t = self.starts[si]
            x = self.X[t:t+self.seq_len, li, :]                     # (S, F)
            y = self.Y[t+self.seq_len:t+self.seq_len+self.horizon, li]  # (H,)
        else:
            t = self.starts[idx]
            x = self.X[t:t+self.seq_len]                            # (S, L, F)
            y = self.Y[t+self.seq_len:t+self.seq_len+self.horizon]  # (H, L)
        
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        
        # 训练时加噪声 (数据增强, 防过拟合)
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        
        return x, y

# GNN 需要 all_counties，其余用 per_county
DS_MODE = 'all_counties' if GNN_ENABLED else 'per_county'

train_ds = OutageDataset(X_train_3d, Y_train_3d, SEQ_LEN, HORIZON,
                         mode=DS_MODE, stride=TRAIN_STRIDE, noise_std=INPUT_NOISE_STD)
val_ds   = OutageDataset(X_val_3d,   Y_val_3d,   SEQ_LEN, HORIZON,
                         mode=DS_MODE, stride=1, noise_std=0.0)  # 验证集不加噪声、stride=1
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f'Mode: {DS_MODE} | Stride: {TRAIN_STRIDE} | Noise: {INPUT_NOISE_STD}')
print(f'Train: {len(train_ds):,} (stride={TRAIN_STRIDE}) | Val: {len(val_ds):,} (stride=1)')
print(f'Batch X: {xb.shape} | Batch Y: {yb.shape}')
print(f'  (去冗余前 train 约 {len(val_ds) * (1-VALIDATION_SPLIT)/VALIDATION_SPLIT:,.0f} 样本, '
      f'现在 {len(train_ds):,}, 减少 {1 - len(train_ds)/(len(val_ds)*(1-VALIDATION_SPLIT)/VALIDATION_SPLIT):.0%})')

---
## §3 模型定义

### 关于 BiLSTM: 用不用?

**结论: 本任务不建议用 BiLSTM，用单向 LSTM 即可。**

原因: BiLSTM 的反向 LSTM 需要看到**未来**的数据来产生 hidden state。
在时序预测中，推理时我们只有过去的数据，没有未来 —— 所以 BiLSTM 的反向部分
在训练时能利用未来信息 (看起来效果好)，但在推理时退化成用全零 hidden state，
等于白训了一半的参数。

> **什么时候该用 BiLSTM?** NLP (整个句子都已知)、分类任务 (输入是完整序列)。
> **时序预测?** 单向 LSTM 就够了。把省下来的参数加到深度或宽度上更划算。

---

### GNN 讲解 (给只学过 CNN 和 RNN 的同学)

**一句话**: CNN 处理网格数据 (图片), RNN 处理序列数据 (时间), GNN 处理**图结构**数据 (节点+边)。

#### 你已经懂的东西

| | CNN | RNN | GNN |
|---|---|---|---|
| **数据结构** | 网格 (像素) | 序列 (时间步) | 图 (节点+边) |
| **核心操作** | 卷积核滑窗 | hidden state 传递 | 邻居信息聚合 |
| **学到什么** | 局部空间模式 | 时间依赖 | **节点间的关系** |

#### 为什么停电预测需要 GNN?

83 个县不是独立的! 一场风暴从西向东扫过 Michigan，Wayne County 先停电，
隔壁 Oakland County 2 小时后也停了。纯 LSTM 把每个县独立预测，看不到这种
**空间传播**。

GNN 做的事: 让每个县"看看邻居的情况"，然后更新自己的预测。

#### GCN 一层的数学 (和 CNN 类比)

```
CNN:  output[i,j] = Σ kernel[m,n] × input[i+m, j+n]     # 对邻居像素加权求和
GCN:  output[county_i] = Σ A[i,j] × W × input[county_j]  # 对邻居县加权求和
```

- `A[i,j]` = 邻接矩阵，定义哪些县是"邻居" (类似 CNN 的卷积核覆盖范围)
- `W` = 可学习权重矩阵 (类似 CNN 的卷积核参数)
- 叠 2 层 GCN = 每个县能看到 2-hop 范围内的所有县 (类似 CNN 增大感受野)

#### 本 notebook 的架构

```
输入: (B, T, 83, F) — B个样本, T个时间步, 83个县, F个特征

Step 1: LSTM (时间维度)
  对每个县独立跑 LSTM → 得到每个县的时间编码 h_lstm[i]
  
Step 2: GCN (空间维度)  
  h_gnn[i] = ReLU(Σ_j A[i,j] × W × h_lstm[j])
  # 每个县聚合邻居的 LSTM 编码
  
Step 3: 融合
  h_final[i] = Gate(h_lstm[i], h_gnn[i])
  # 用门控决定时间信息和空间信息的比例
  
Step 4: 预测
  pred[i] = Linear(h_final[i])
```

In [ ]:
# §3.1 三个基础模型

class DeepLSTM(nn.Module):
    """
    加强版多层 LSTM
    - LayerNorm 稳定深层训练
    - Residual connection (层间跳跃连接)
    - 简洁 MLP head (防过拟合)
    """
    def __init__(self, input_dim, hidden_dim, num_layers, horizon,
                 dropout=0.2, use_gru=False, bidirectional=False):
        super().__init__()
        ndir = 2 if bidirectional else 1
        self.hidden_dim = hidden_dim
        self.ndir = ndir
        
        RNN = nn.GRU if use_gru else nn.LSTM
        self.rnn = RNN(input_dim, hidden_dim, num_layers, batch_first=True,
                       dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)
        
        self.residual_proj = nn.Linear(input_dim, hidden_dim * ndir)
        self.norm = nn.LayerNorm(hidden_dim * ndir)
        self.drop = nn.Dropout(dropout)
        
        # 简化 head: 1 层隐藏 + 输出 (之前 3 层太重, 过拟合)
        hdim = hidden_dim * ndir
        self.head = nn.Sequential(
            nn.Linear(hdim, hdim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hdim // 2, horizon),
        )

    def forward(self, x):
        """x: (B, T, F) → (B, horizon)"""
        rnn_out, _ = self.rnn(x)              # (B, T, H*ndir)
        last = rnn_out[:, -1, :]              # (B, H*ndir)
        res = self.residual_proj(x[:, -1, :]) # (B, H*ndir)
        last = self.drop(self.norm(last + res))
        return self.head(last)


class Seq2SeqAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, horizon, dropout=0.2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.attn_We = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attn_Wd = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attn_v  = nn.Linear(hidden_dim, 1, bias=False)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon))
    def attention(self, enc_out, h):
        e = torch.tanh(self.attn_We(enc_out) + self.attn_Wd(h).unsqueeze(1))
        w = torch.softmax(self.attn_v(e).squeeze(-1), dim=1)
        return torch.bmm(w.unsqueeze(1), enc_out).squeeze(1), w
    def forward(self, x, return_attention=False):
        enc_out, (h_n, _) = self.encoder(x)
        ctx, aw = self.attention(enc_out, h_n[-1])
        out = self.head(torch.cat([h_n[-1], ctx], dim=1))
        return (out, aw) if return_attention else out

class TransformerForecaster(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, num_layers, horizon,
                 dim_ff=512, dropout=0.1, max_len=200, causal=True):
        super().__init__()
        self.causal = causal
        self.proj = nn.Linear(input_dim, d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div[:d_model//2]) if d_model%2 else torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.drop = nn.Dropout(dropout)
        el = nn.TransformerEncoderLayer(d_model, n_heads, dim_ff, dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(el, num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, horizon))
    def forward(self, x):
        B,T,_ = x.shape
        x = self.drop(self.proj(x) + self.pe[:,:T])
        mask = torch.triu(torch.ones(T,T,device=x.device),1).bool() if self.causal else None
        return self.head(self.encoder(x, mask=mask)[:,-1,:])

print('✓ DeepLSTM (v2: 简化 head + dropout) / Seq2Seq / Transformer')

In [ ]:
# §3.2 GNN 空间模块

# ======================== GCN 层 ========================
class GCNLayer(nn.Module):
    """
    一层 Graph Convolutional Network
    
    类比 CNN:
      CNN:  对邻居像素做加权求和 (卷积核固定 3×3)
      GCN:  对邻居节点做加权求和 (邻接矩阵定义谁是邻居)
    
    公式: H' = ReLU(A_norm @ H @ W + b)
      A_norm: 归一化邻接矩阵 (83×83), 预计算
      H: 节点特征 (83×in_dim)
      W: 可学习权重 (in_dim×out_dim)
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.b = nn.Parameter(torch.zeros(out_dim))
    
    def forward(self, x, A):
        """
        x: (B, 83, in_dim) — 每个县的特征向量
        A: (83, 83)         — 归一化邻接矩阵
        """
        h = self.W(x)                # (B, 83, out_dim) — 线性变换
        h = torch.matmul(A, h)       # (B, 83, out_dim) — 邻居聚合 (图卷积核心!)
        return torch.relu(h + self.b)


# ======================== LSTM + GNN 融合模型 ========================
class SpatialWrapper(nn.Module):
    """
    在任何时序 backbone 上叠加 GNN 空间模块。
    
    架构:
      (B, T, L, F)
        ↓ 每县独立 LSTM 编码
      (B, L, H_lstm)
        ↓ 2 层 GCN 聚合邻居信息  
      (B, L, H_gnn)
        ↓ 门控融合 LSTM + GCN
      (B, L, H_fused)
        ↓ MLP 预测头
      (B, L, horizon)
    """
    def __init__(self, backbone, hidden_dim, horizon, n_counties,
                 gnn_hidden=64, fusion='gate', dropout=0.2):
        super().__init__()
        self.backbone = backbone  # DeepLSTM 的 rnn 部分
        self.n_counties = n_counties
        self.fusion = fusion
        
        # 从 backbone 获取 rnn 的输出维度
        rnn_out_dim = hidden_dim * (2 if hasattr(backbone, 'ndir') and backbone.ndir == 2 else 1)
        
        # 2 层 GCN (2-hop 邻居)
        self.gcn1 = GCNLayer(rnn_out_dim, gnn_hidden)
        self.gcn2 = GCNLayer(gnn_hidden, gnn_hidden)
        
        # 融合方式
        if fusion == 'concat':
            head_in = rnn_out_dim + gnn_hidden
        elif fusion == 'gate':
            # 门控: g = sigmoid(W[h_lstm; h_gnn]) → h = g⊙h_lstm + (1-g)⊙proj(h_gnn)
            self.gate_fc = nn.Linear(rnn_out_dim + gnn_hidden, rnn_out_dim)
            self.gnn_proj = nn.Linear(gnn_hidden, rnn_out_dim)
            head_in = rnn_out_dim
        else:  # add
            self.gnn_proj = nn.Linear(gnn_hidden, rnn_out_dim)
            head_in = rnn_out_dim
        
        self.head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Linear(head_in, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, horizon),
        )
    
    def forward(self, x, adj):
        """
        x: (B, T, L, F) — 所有县的时间序列
        adj: (L, L) — 归一化邻接矩阵
        returns: (B, L, horizon)
        """
        B, T, L_loc, F = x.shape
        
        # Step 1: 每县独立过 LSTM (共享参数)
        x_flat = x.permute(0, 2, 1, 3).reshape(B * L_loc, T, F)  # (B*L, T, F)
        rnn_out, _ = self.backbone.rnn(x_flat)       # (B*L, T, H)
        h_lstm = rnn_out[:, -1, :].reshape(B, L_loc, -1)  # (B, L, H)
        
        # Step 2: 2 层 GCN 空间传播
        h_gnn = self.gcn1(h_lstm, adj)               # (B, L, gnn_h) — 1-hop 邻居
        h_gnn = self.gcn2(h_gnn, adj)                # (B, L, gnn_h) — 2-hop 邻居
        
        # Step 3: 融合
        if self.fusion == 'concat':
            h = torch.cat([h_lstm, h_gnn], dim=-1)   # (B, L, H+gnn_h)
        elif self.fusion == 'gate':
            g = torch.sigmoid(self.gate_fc(torch.cat([h_lstm, h_gnn], dim=-1)))  # (B, L, H)
            h = g * h_lstm + (1 - g) * self.gnn_proj(h_gnn)   # (B, L, H)
        else:
            h = h_lstm + self.gnn_proj(h_gnn)
        
        # Step 4: 预测
        return self.head(h)  # (B, L, horizon)


def build_adj(locations, k=5):
    """
    构建归一化邻接矩阵 — 基于 Michigan 各县真实地理坐标 (US Census 2020 人口加权质心)。

    策略:
      1. FIPS → 经纬度
      2. Haversine 距离算各县间地理距离
      3. 每个县取 k 个最近邻, 边权重 = exp(-dist / sigma) (高斯核, sigma=邻居距离中位数)
      4. 对称化 + self-loop + D^{-1/2} A D^{-1/2} 归一化 (标准 GCN 归一化)
    """
    # ---- Michigan 83 县质心 (US Census 2020 Centers of Population) ----
    MI_COUNTY_CENTROIDS = {
        "26001": (44.6681, -83.4656), "26003": (46.3845, -86.7118),
        "26005": (42.6084, -85.8710), "26007": (45.0506, -83.5029),
        "26009": (44.9783, -85.1986), "26011": (44.0445, -83.8845),
        "26013": (46.7665, -88.4500), "26015": (42.6185, -85.3494),
        "26017": (43.6300, -83.9292), "26019": (44.6518, -86.0114),
        "26021": (41.9857, -86.4064), "26023": (41.9292, -85.0307),
        "26025": (42.2896, -85.1060), "26027": (41.8923, -86.0405),
        "26029": (45.2445, -85.1110), "26031": (45.5110, -84.5211),
        "26033": (46.3656, -84.3963), "26035": (43.9444, -84.8257),
        "26037": (42.8863, -84.5543), "26039": (44.6571, -84.6690),
        "26041": (45.8019, -87.0658), "26043": (45.8330, -88.0140),
        "26045": (42.6451, -84.7396), "26047": (45.4320, -84.9104),
        "26049": (43.0024, -83.6938), "26051": (43.9680, -84.4398),
        "26053": (46.4464, -89.9930), "26055": (44.7134, -85.6130),
        "26057": (43.3441, -84.6313), "26059": (41.9169, -84.6052),
        "26061": (47.1273, -88.5418), "26063": (43.8324, -83.0630),
        "26065": (42.6884, -84.4863), "26067": (42.9570, -85.0867),
        "26069": (44.3570, -83.5324), "26071": (46.0996, -88.5376),
        "26073": (43.6085, -84.8035), "26075": (42.2418, -84.4066),
        "26077": (42.2625, -85.5864), "26079": (44.7206, -85.1650),
        "26081": (42.9618, -85.6256), "26083": (47.3361, -88.2893),
        "26085": (43.9572, -85.7994), "26087": (43.0582, -83.2494),
        "26089": (44.9070, -85.7301), "26091": (41.9261, -84.0528),
        "26093": (42.5750, -83.8565), "26095": (46.3357, -85.5614),
        "26097": (45.9973, -84.8813), "26099": (42.5891, -82.9579),
        "26101": (44.3063, -86.1853), "26103": (46.4867, -87.4928),
        "26105": (43.9746, -86.3467), "26107": (43.6509, -85.3822),
        "26109": (45.3421, -87.5798), "26111": (43.6433, -84.2964),
        "26113": (44.3077, -85.1799), "26115": (41.9017, -83.4864),
        "26117": (43.2836, -85.1727), "26119": (44.9974, -84.1259),
        "26121": (43.2438, -86.2149), "26123": (43.4662, -85.8113),
        "26125": (42.5869, -83.3140), "26127": (43.6327, -86.3058),
        "26129": (44.2866, -84.1274), "26131": (46.7189, -89.2882),
        "26133": (43.9654, -85.3578), "26135": (44.7015, -84.1484),
        "26137": (45.0069, -84.6668), "26139": (42.9111, -86.0159),
        "26141": (45.3570, -83.8682), "26143": (44.3643, -84.6501),
        "26145": (43.3991, -83.9964), "26147": (42.9122, -82.5635),
        "26149": (41.8875, -85.5269), "26151": (43.3652, -82.7830),
        "26153": (46.0061, -86.2268), "26155": (42.9427, -84.1479),
        "26157": (43.4299, -83.4339), "26159": (42.2602, -86.0013),
        "26161": (42.2576, -83.7269), "26163": (42.3295, -83.2263),
        "26165": (44.3097, -85.4866),
    }

    n = len(locations)
    loc_strs = [str(loc) for loc in locations]

    # 经纬度 → numpy array
    coords = np.array([MI_COUNTY_CENTROIDS[fips] for fips in loc_strs])  # (n, 2)
    lat_rad = np.radians(coords[:, 0])
    lon_rad = np.radians(coords[:, 1])

    # Haversine 距离矩阵 (km)
    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]
    a = (np.sin(dlat / 2) ** 2 +
         np.cos(lat_rad[:, None]) * np.cos(lat_rad[None, :]) * np.sin(dlon / 2) ** 2)
    dist = 6371.0 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))  # (n, n) km

    # KNN: 每个节点取 k 个最近邻 (排除自身)
    A = np.zeros((n, n))
    for i in range(n):
        d_i = dist[i].copy()
        d_i[i] = np.inf
        neighbors = np.argsort(d_i)[:k]
        A[i, neighbors] = 1.0

    # 对称化: i→j 或 j→i 是邻居 → 双向连接
    A = np.maximum(A, A.T)

    # 高斯核权重: w_ij = exp(-dist_ij / sigma), sigma = 邻居距离中位数
    neighbor_dists = dist[A > 0]
    sigma = np.median(neighbor_dists) if len(neighbor_dists) > 0 else 1.0
    W = np.exp(-dist / sigma)
    A = A * W  # 只保留邻居边, 带距离衰减权重

    # 加 self-loop
    A = A + np.eye(n)

    # D^{-1/2} A D^{-1/2} 对称归一化 (标准 GCN)
    D = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(D, 1e-12)))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    print(f"  build_adj: {n} counties, k={k}, σ={sigma:.1f}km, "
          f"edges={int((A > np.eye(n)).sum())} (excl self-loop)")
    return torch.tensor(A_norm, dtype=torch.float32)


# ======================== Two-Stage & Ensemble (占位) ========================
class TwoStageWrapper:
    def __init__(self, cls_model, reg_model, threshold=0.5):
        self.cls, self.reg, self.threshold = cls_model, reg_model, threshold
    def predict(self, x):
        with torch.no_grad():
            return (torch.sigmoid(self.cls(x)) > self.threshold).float() * torch.clamp(self.reg(x), min=0)

class EnsemblePredictor:
    def __init__(self, method='stacking'):
        self.method = method
        self.preds, self.rmses, self.county_rmses = {}, {}, {}
    def add(self, name, vp, vt, locs):
        self.preds[name] = vp
        cr = {locs[i]: np.sqrt(np.mean((vt[:,i]-vp[:,i])**2)) for i in range(len(locs))}
        self.county_rmses[name] = cr; self.rmses[name] = np.mean(list(cr.values()))
    def fit(self, vt, locs):
        pass
    def predict(self, tp, locs):
        return np.mean(list(tp.values()), axis=0)


print('✓ GCN / SpatialWrapper / TwoStage / Ensemble 已定义')

In [ ]:
# §3.3 自动实例化
INPUT_DIM = len(feature_cols)

# ---- 构建 backbone ----
if MODEL_TYPE == 'lstm':
    backbone = DeepLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON,
                        DROPOUT, USE_GRU, BIDIRECTIONAL)
elif MODEL_TYPE == 'seq2seq':
    backbone = Seq2SeqAttention(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON, DROPOUT)
elif MODEL_TYPE == 'transformer':
    backbone = TransformerForecaster(INPUT_DIM, HIDDEN_DIM, N_HEADS, NUM_LAYERS, HORIZON,
                                     DIM_FEEDFORWARD, DROPOUT, causal=USE_CAUSAL_MASK)
else:
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

# ---- 叠加 GNN? ----
if GNN_ENABLED:
    adj_matrix = build_adj(locations, GNN_K).to(DEVICE)
    model = SpatialWrapper(backbone, HIDDEN_DIM, HORIZON, L,
                           GNN_HIDDEN, SPATIAL_FUSION, DROPOUT).to(DEVICE)
    print(f'Model: {MODEL_TYPE} + GNN ({SPATIAL_FUSION})')
else:
    model = backbone.to(DEVICE)
    adj_matrix = None
    print(f'Model: {MODEL_TYPE}')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Params: {n_params:,} | Device: {DEVICE}')
print(model)

---
## §4 损失函数

In [ ]:
# §4 损失函数
class WeightedMSE(nn.Module):
    def __init__(self, threshold=500, weight=5.0):
        super().__init__()
        self.threshold, self.weight = threshold, weight
    def forward(self, pred, target):
        w = torch.where(target > self.threshold, self.weight, 1.0)
        return (w * (pred - target)**2).mean()

if LOSS_FN == 'huber':
    criterion = nn.HuberLoss(delta=HUBER_DELTA)
elif LOSS_FN == 'weighted_mse':
    criterion = WeightedMSE(EXTREME_THRESHOLD, EXTREME_WEIGHT)
else:
    criterion = nn.MSELoss()

# Two-Stage 的分类 loss
cls_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([CLS_POS_WEIGHT]).to(DEVICE)) if TWO_STAGE_ENABLED else None

print(f'Loss: {criterion}')
if cls_criterion: print(f'CLS Loss: {cls_criterion}')

---
## §5 wandb

In [ ]:
# §5 wandb 初始化
WANDB_ENABLED = False

try:
    import wandb

    # ---- 获取配置: 优先 .env, 其次 Colab secrets, 最后手动填 ----
    api_key, entity, user = None, None, 'unknown'
    
    # 方式 1: .env 文件 (本地)
    try:
        from dotenv import load_dotenv; load_dotenv()
        api_key = os.getenv('WANDB_API_KEY')
        entity  = os.getenv('WANDB_ENTITY')
        user    = os.getenv('WANDB_USERNAME', 'unknown')
    except ImportError:
        pass
    
    # 方式 2: Colab userdata (Colab 左侧 🔑 Secrets 面板)
    if not api_key and IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get('WANDB_API_KEY')
            entity  = userdata.get('WANDB_ENTITY')
            user    = userdata.get('WANDB_USERNAME')
        except Exception:
            pass
    
    # 方式 3: 直接在这里填 (取消注释)
    # api_key = "your_wandb_api_key"
    # entity  = "your_team_name"
    # user    = "your_name"

    if api_key and entity and entity != 'your_team_name':
        wandb.login(key=api_key)
        run_name = f'{user}_{MODEL_NAME}_{EXPERIMENT_TAG}_{datetime.now().strftime("%m%d_%H%M")}'
        wandb.init(project='MLPS-Power-Outage', entity=entity, name=run_name,
                   tags=['phase2', MODEL_TYPE, EXPERIMENT_TAG],
                   config={k: v for k, v in globals().items()
                           if isinstance(v, (int, float, str, bool, list, tuple)) and k.isupper()})
        WANDB_ENABLED = True
        print(f'✓ wandb: {run_name}')
    else:
        print('⚠ wandb 未配置 — 不影响训练, 跳过')
        if IN_COLAB:
            print('  Colab 配置方法: 左侧 🔑 图标 → Add new secret → 添加 WANDB_API_KEY / WANDB_ENTITY')
            print('  或直接在本 cell 取消注释 "方式 3" 手动填入')

except Exception as e:
    print(f'⚠ wandb 不可用: {e} — 不影响训练, 跳过')

---
## §6 训练循环

如果 `ENSEMBLE_ENABLED=True`，跳过此节直接到 §7。

In [ ]:
# §6.1 优化器 + 调度器
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
if LR_SCHEDULER == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, eta_min=LEARNING_RATE*0.01)
elif LR_SCHEDULER == 'step':
    scheduler = optim.lr_scheduler.StepLR(optimizer, 20, 0.5)
elif LR_SCHEDULER == 'plateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', 0.5, patience=5)
else:
    scheduler = None
print(f'Optimizer: AdamW | Scheduler: {LR_SCHEDULER}')

In [ ]:
# §6.2 训练 + 验证
if ENSEMBLE_ENABLED:
    print('ENSEMBLE_ENABLED=True, 跳过训练 → 直接到 §7')
else:
    best_val_rmse, best_epoch, patience_ctr = float('inf'), 0, 0
    history = {'train_loss':[], 'val_loss':[], 'val_rmse':[], 'lr':[]}
    t_start = time.time()

    print(f'\n{"="*60}')
    print(f'Training: {MODEL_NAME} | {EPOCHS} epochs | {DEVICE}')
    print(f'{"="*60}\n')

    for epoch in range(1, EPOCHS + 1):
        # ---- Train ----
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            # GNN 模型需要传 adj_matrix
            pred = model(xb, adj_matrix) if GNN_ENABLED else model(xb)
            # Two-Stage 分类阶段: target 是 (y>0).float()
            if TWO_STAGE_ENABLED and cls_criterion is not None:
                loss = cls_criterion(pred, (yb > 0).float())
            else:
                loss = criterion(pred, yb)
            loss.backward()
            if GRADIENT_CLIP: nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            losses.append(loss.item())
        train_loss = np.mean(losses)

        # ---- Validate ----
        model.eval()
        vl, vp, vt = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb, adj_matrix) if GNN_ENABLED else model(xb)
                if TWO_STAGE_ENABLED and cls_criterion is not None:
                    vl.append(cls_criterion(pred, (yb>0).float()).item())
                else:
                    vl.append(criterion(pred, yb).item())
                vp.append(pred.cpu().numpy()); vt.append(yb.cpu().numpy())
        val_loss = np.mean(vl)
        all_p, all_t = np.concatenate(vp), np.concatenate(vt)
        val_rmse = np.sqrt(np.mean((all_p - all_t)**2))

        # ---- LR schedule ----
        cur_lr = optimizer.param_groups[0]['lr']
        if scheduler:
            scheduler.step(val_rmse) if LR_SCHEDULER == 'plateau' else scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)
        history['lr'].append(cur_lr)

        if WANDB_ENABLED:
            wandb.log({'epoch': epoch, 'train/loss': train_loss,
                       'val/loss': val_loss, 'val/rmse': val_rmse, 'lr': cur_lr})

        if val_rmse < best_val_rmse:
            best_val_rmse, best_epoch, patience_ctr = val_rmse, epoch, 0
            os.makedirs(RESULTS_DIR, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'))
        else:
            patience_ctr += 1

        if epoch % 5 == 0 or epoch == 1 or patience_ctr == 0:
            m = '★' if patience_ctr == 0 else ' '
            print(f'  E{epoch:3d}/{EPOCHS} | TrLoss:{train_loss:.4f} ValRMSE:{val_rmse:.4f} '
                  f'Best:{best_val_rmse:.4f}@{best_epoch} LR:{cur_lr:.6f} {m}')

        if patience_ctr >= EARLY_STOPPING_PATIENCE:
            print(f'\n  ⏹ Early stop @ epoch {epoch}')
            break

    total_time = time.time() - t_start
    print(f'\n✓ {total_time:.0f}s | Best: {best_val_rmse:.4f} @ epoch {best_epoch}')

In [ ]:
# §6.3 训练曲线
if not ENSEMBLE_ENABLED:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].axvline(best_epoch-1, c='r', ls='--', alpha=.5); axes[0].legend()
    axes[0].set(xlabel='Epoch', ylabel='Loss', yscale='log', title='Loss')
    axes[1].plot(history['val_rmse'], c='#e74c3c')
    axes[1].axhline(best_val_rmse, c='g', ls='--', alpha=.5, label=f'Best={best_val_rmse:.2f}')
    axes[1].legend(); axes[1].set(xlabel='Epoch', ylabel='RMSE', title='Val RMSE')
    axes[2].plot(history['lr'], c='#3498db')
    axes[2].set(xlabel='Epoch', ylabel='LR', yscale='log', title='Learning Rate')
    plt.suptitle(f'{MODEL_NAME} ({EXPERIMENT_TAG})', fontweight='bold'); plt.tight_layout(); plt.show()
    if WANDB_ENABLED: wandb.log({'charts/curves': wandb.Image(fig)})

---
## §7 统一评估 (县均 RMSE)

In [ ]:
# §7.1 评估函数
def evaluate(y_true, y_pred, locs, name='Model'):
    cr = {locs[i]: np.sqrt(np.mean((y_true[:,i]-y_pred[:,i])**2)) for i in range(len(locs))}
    avg = np.mean(list(cr.values()))
    print(f'=== {name}: RMSE={avg:.4f} ===')
    for loc, r in sorted(cr.items(), key=lambda x: -x[1])[:5]:
        print(f'  {loc}: {r:.4f}')
    return avg, cr

def evaluate_horizons(y_true, y_pred, locs, name):
    res = {}
    if y_true.shape[0] >= 24:
        r24, _ = evaluate(y_true[:24], y_pred[:24], locs, f'{name}_24h')
        res[f'{name}_24h'] = r24
    if y_true.shape[0] >= 48:
        r48, _ = evaluate(y_true[:48], y_pred[:48], locs, f'{name}_48h')
        res[f'{name}_48h'] = r48
    rf, cr = evaluate(y_true, y_pred, locs, f'{name}_full')
    res[f'{name}_full'] = rf
    return res, cr

print('✓ evaluate / evaluate_horizons')

In [ ]:
# §7.2 生成验证集预测
if not ENSEMBLE_ENABLED:
    model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'),
                                     map_location=DEVICE, weights_only=True))
    model.eval()

    Y_pred_val = np.zeros_like(Y_val_3d)
    counts = np.zeros_like(Y_val_3d)

    with torch.no_grad():
        X_full = np.concatenate([X_train_3d[-SEQ_LEN:], X_val_3d], axis=0)
        
        if GNN_ENABLED:
            for t in range(0, X_full.shape[0]-SEQ_LEN-HORIZON+1, HORIZON):
                xin = torch.tensor(X_full[t:t+SEQ_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                pred = model(xin, adj_matrix).cpu().numpy().squeeze(0)
                vs = t + SEQ_LEN - SEQ_LEN
                ve = min(vs + HORIZON, Y_val_3d.shape[0])
                nf = ve - max(vs, 0)
                if vs >= 0 and nf > 0:
                    Y_pred_val[vs:ve] += pred[:nf]
                    counts[vs:ve] += 1
        else:
            for li in range(L):
                for t in range(0, X_full.shape[0]-SEQ_LEN-HORIZON+1, HORIZON):
                    xin = torch.tensor(X_full[t:t+SEQ_LEN, li], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                    pred = model(xin).cpu().numpy().flatten()
                    vs = t + SEQ_LEN - SEQ_LEN
                    ve = min(vs + HORIZON, Y_val_3d.shape[0])
                    nf = ve - max(vs, 0)
                    if vs >= 0 and nf > 0:
                        Y_pred_val[vs:ve, li] += pred[:nf]
                        counts[vs:ve, li] += 1

    counts[counts == 0] = 1
    Y_pred_val = Y_pred_val / counts
    
    # 如果训练用了 log1p, 还原回原始空间
    if LOG_TRANSFORM_Y:
        Y_pred_val = np.expm1(np.clip(Y_pred_val, 0, 20))  # clip 防止 exp 爆炸
        print(f'✓ log1p → expm1 还原完成')
    
    Y_pred_val = np.clip(Y_pred_val, 0, None)
    
    os.makedirs(RESULTS_DIR, exist_ok=True)
    np.save(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_val_pred.npy'), Y_pred_val)
    print(f'Val pred: {Y_pred_val.shape} | range [{Y_pred_val.min():.1f}, {Y_pred_val.max():.1f}]')

In [ ]:
# §7.3 评估 + 排行榜对比
BASELINES = {
    'SARIMAX_24h': 27.91, 'SARIMAX_48h': 20.13,
    'HistGBM_FE_24h': 54.37, 'HistGBM_FE_48h': 44.02,
    'HistAvg_24h': 93.03, 'HistAvg_48h': 73.55,
    'Seq2Seq_v1_24h': 100.69, 'Seq2Seq_v1_48h': 109.38,
    'Persistence_24h': 129.17, 'Persistence_48h': 117.76,
}

# 评估用原始空间的 Y (不是 log 空间)
Y_eval_true = Y_val_3d_raw if LOG_TRANSFORM_Y else Y_val_3d

if ENSEMBLE_ENABLED:
    print('⚠ Ensemble: 请取消注释加载各模型预测')
    model_results = {}
else:
    model_results, model_county_rmses = evaluate_horizons(Y_eval_true, Y_pred_val, locations, MODEL_NAME)

all_results = {**BASELINES, **model_results}
print(f'\n{"="*70}')
print('RMSE 排行榜 (原始空间)')
print(f'{"="*70}')
for h in ['24h', '48h']:
    print(f'\n--- {h} ---')
    hr = {k:v for k,v in all_results.items() if h in k}
    for name, rmse in sorted(hr.items(), key=lambda x: x[1]):
        tag = '  ◀ YOU' if MODEL_NAME in name else ''
        print(f'  {name:25s}: {rmse:10.4f}{tag}')

if WANDB_ENABLED:
    for k,v in model_results.items(): wandb.summary[f'eval/{k}'] = v

In [ ]:
# §7.4 排行榜可视化
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, h in zip(axes, ['24h', '48h']):
    hr = sorted([(k,v) for k,v in all_results.items() if h in k], key=lambda x: x[1])
    if not hr: continue
    names, rmses = zip(*hr)
    colors = ['#e74c3c' if MODEL_NAME in n else '#2196F3' if 'SARIMAX' in n
              else '#4CAF50' if 'GBM' in n else '#FF9800' if 'Seq2Seq' in n else '#bdbdbd' for n in names]
    bars = ax.barh(range(len(names)), rmses, color=colors, alpha=.85)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel('县均 RMSE'); ax.set_title(f'{h}', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    for i, (b, v) in enumerate(zip(bars, rmses)):
        ax.text(v + max(rmses)*.01, i, f'{v:.2f}', va='center', fontsize=9)
plt.suptitle(f'排行榜 — {MODEL_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
if WANDB_ENABLED: wandb.log({'charts/leaderboard': wandb.Image(fig)})

---
## §P2: Phase 2 — 加入 GNN 空间模块

上面的 Phase 1 结果是纯 LSTM 的。现在开启 GNN，重新训练，看空间信息能提升多少。

In [ ]:
# §P2.1: 重新构建 GNN 版本的 DataLoader + 模型

print("=" * 60)
print("Phase 2: Deep LSTM + GNN")
print("=" * 60)

# 切换到 all_counties 模式
GNN_ENABLED_P2 = True
DS_MODE_P2 = 'all_counties'

train_ds_gnn = OutageDataset(X_train_3d, Y_train_3d, SEQ_LEN, HORIZON, mode=DS_MODE_P2)
val_ds_gnn   = OutageDataset(X_val_3d,   Y_val_3d,   SEQ_LEN, HORIZON, mode=DS_MODE_P2)
train_loader_gnn = DataLoader(train_ds_gnn, batch_size=min(BATCH_SIZE, 16), shuffle=True, num_workers=0)
val_loader_gnn   = DataLoader(val_ds_gnn,   batch_size=min(BATCH_SIZE, 32), shuffle=False, num_workers=0)

xb_g, yb_g = next(iter(train_loader_gnn))
print(f"GNN DataLoader: X={xb_g.shape} Y={yb_g.shape}")
print(f"  X = (batch, seq_len={SEQ_LEN}, counties={L}, features={len(feature_cols)})")
print(f"  Y = (batch, horizon={HORIZON}, counties={L})")

# 构建邻接矩阵
adj_matrix_gnn = build_adj(locations, GNN_K).to(DEVICE)
print(f"\n邻接矩阵: {adj_matrix_gnn.shape}")

# 用 Phase 1 训好的 LSTM 作为 backbone (迁移学习)
backbone_gnn = DeepLSTM(
    INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON,
    DROPOUT, USE_GRU, BIDIRECTIONAL
).to(DEVICE)

# 加载 Phase 1 的权重到 backbone
phase1_ckpt = torch.load(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'),
                          map_location=DEVICE, weights_only=True)
backbone_gnn.load_state_dict(phase1_ckpt)
print("✓ 加载 Phase 1 LSTM 权重 (迁移学习)")

# 构建 GNN 模型
model_gnn = SpatialWrapper(
    backbone_gnn, HIDDEN_DIM, HORIZON, L,
    gnn_hidden=GNN_HIDDEN, fusion=SPATIAL_FUSION, dropout=DROPOUT
).to(DEVICE)

n_params_gnn = sum(p.numel() for p in model_gnn.parameters() if p.requires_grad)
print(f"\nLSTM+GNN 参数量: {n_params_gnn:,} (vs Phase 1: {n_params:,}, +{n_params_gnn-n_params:,})")

In [ ]:
# §P2.2: 训练 LSTM+GNN

GNN_MODEL_NAME = "DeepLSTM_GNN"
GNN_EPOCHS = 60
GNN_LR = 3e-4  # GNN 部分用更小的学习率 (backbone 已预训练)

optimizer_gnn = optim.AdamW([
    {'params': model_gnn.backbone.parameters(), 'lr': GNN_LR * 0.1},  # backbone 低学习率 (微调)
    {'params': model_gnn.gcn1.parameters(), 'lr': GNN_LR},
    {'params': model_gnn.gcn2.parameters(), 'lr': GNN_LR},
    {'params': model_gnn.head.parameters(), 'lr': GNN_LR},
] + ([{'params': model_gnn.gate_fc.parameters(), 'lr': GNN_LR},
      {'params': model_gnn.gnn_proj.parameters(), 'lr': GNN_LR}]
     if hasattr(model_gnn, 'gate_fc') else []),
    weight_decay=WEIGHT_DECAY)

scheduler_gnn = optim.lr_scheduler.CosineAnnealingLR(optimizer_gnn, GNN_EPOCHS, eta_min=GNN_LR*0.01)

best_gnn_rmse, best_gnn_epoch, patience_gnn = float('inf'), 0, 0
history_gnn = {'train_loss':[], 'val_loss':[], 'val_rmse':[], 'lr':[]}
t_start_gnn = time.time()

print(f"Training LSTM+GNN | {GNN_EPOCHS} epochs\n")

for epoch in range(1, GNN_EPOCHS + 1):
    model_gnn.train()
    losses = []
    for xb, yb in train_loader_gnn:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)     # xb: (B,T,L,F), yb: (B,H,L)
        optimizer_gnn.zero_grad()
        pred = model_gnn(xb, adj_matrix_gnn)        # (B, L, H)
        pred = pred.permute(0, 2, 1)                 # (B, H, L) — 对齐 yb
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model_gnn.parameters(), GRADIENT_CLIP)
        optimizer_gnn.step()
        losses.append(loss.item())
    train_loss = np.mean(losses)

    model_gnn.eval()
    vl, vp, vt_list = [], [], []
    with torch.no_grad():
        for xb, yb in val_loader_gnn:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model_gnn(xb, adj_matrix_gnn).permute(0, 2, 1)
            vl.append(criterion(pred, yb).item())
            vp.append(pred.cpu().numpy()); vt_list.append(yb.cpu().numpy())
    val_loss = np.mean(vl)
    all_p, all_t = np.concatenate(vp), np.concatenate(vt_list)
    val_rmse = np.sqrt(np.mean((all_p - all_t)**2))

    cur_lr = optimizer_gnn.param_groups[0]['lr']
    scheduler_gnn.step()

    history_gnn['train_loss'].append(train_loss)
    history_gnn['val_loss'].append(val_loss)
    history_gnn['val_rmse'].append(val_rmse)
    history_gnn['lr'].append(cur_lr)

    if WANDB_ENABLED:
        wandb.log({'epoch_gnn': epoch, 'gnn/train_loss': train_loss,
                   'gnn/val_loss': val_loss, 'gnn/val_rmse': val_rmse})

    if val_rmse < best_gnn_rmse:
        best_gnn_rmse, best_gnn_epoch, patience_gnn = val_rmse, epoch, 0
        torch.save(model_gnn.state_dict(), os.path.join(RESULTS_DIR, f'{GNN_MODEL_NAME}_best.pt'))
    else:
        patience_gnn += 1

    if epoch % 5 == 0 or epoch == 1 or patience_gnn == 0:
        m = '★' if patience_gnn == 0 else ' '
        print(f'  E{epoch:3d}/{GNN_EPOCHS} | Loss:{train_loss:.4f} ValRMSE:{val_rmse:.4f} '
              f'Best:{best_gnn_rmse:.4f}@{best_gnn_epoch} {m}')

    if patience_gnn >= EARLY_STOPPING_PATIENCE:
        print(f'\n  ⏹ Early stop @ epoch {epoch}')
        break

total_gnn_time = time.time() - t_start_gnn
print(f'\n✓ GNN训练完成 {total_gnn_time:.0f}s | Best: {best_gnn_rmse:.4f} @ epoch {best_gnn_epoch}')

In [ ]:
# §P2.3: GNN 验证集预测 + 评估

model_gnn.load_state_dict(torch.load(os.path.join(RESULTS_DIR, f'{GNN_MODEL_NAME}_best.pt'),
                                      map_location=DEVICE, weights_only=True))
model_gnn.eval()

Y_pred_gnn = np.zeros_like(Y_val_3d)  # (T_val, L)
counts_gnn = np.zeros_like(Y_val_3d)

with torch.no_grad():
    X_full = np.concatenate([X_train_3d[-SEQ_LEN:], X_val_3d], axis=0)  # (SEQ+T_val, L, F)
    
    for t in range(0, X_full.shape[0] - SEQ_LEN - HORIZON + 1, HORIZON):
        xin = torch.tensor(X_full[t:t+SEQ_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
        pred = model_gnn(xin, adj_matrix_gnn)  # (1, L, HORIZON)
        pred = pred.squeeze(0).permute(1, 0).cpu().numpy()  # (HORIZON, L)
        
        vs = t
        ve = min(vs + HORIZON, Y_val_3d.shape[0])
        nf = ve - max(vs, 0)
        if vs >= 0 and nf > 0:
            Y_pred_gnn[vs:ve] += pred[:nf]
            counts_gnn[vs:ve] += 1

counts_gnn[counts_gnn == 0] = 1
Y_pred_gnn = Y_pred_gnn / counts_gnn

# log1p → expm1 还原
if LOG_TRANSFORM_Y:
    Y_pred_gnn = np.expm1(np.clip(Y_pred_gnn, 0, 20))
    print('✓ GNN 预测: log1p → expm1 还原完成')

Y_pred_gnn = np.clip(Y_pred_gnn, 0, None)

# 保存
np.save(os.path.join(RESULTS_DIR, f'{GNN_MODEL_NAME}_val_pred.npy'), Y_pred_gnn)

# 评估 (原始空间)
Y_eval_true = Y_val_3d_raw if LOG_TRANSFORM_Y else Y_val_3d
gnn_results, gnn_county_rmses = evaluate_horizons(Y_eval_true, Y_pred_gnn, locations, GNN_MODEL_NAME)
print()

In [ ]:
# §P2.4: Phase 1 vs Phase 2 对比

print("=" * 70)
print("Phase 1 (LSTM only) vs Phase 2 (LSTM + GNN) 对比")
print("=" * 70)

# 合并所有结果
all_final = {**BASELINES, **model_results, **gnn_results}

for h in ['24h', '48h']:
    print(f"\n--- {h} ---")
    hr = {k:v for k,v in all_final.items() if h in k}
    for name, rmse in sorted(hr.items(), key=lambda x: x[1]):
        tag = ''
        if 'GNN' in name: tag = '  ◀ LSTM+GNN'
        elif MODEL_NAME in name: tag = '  ◀ LSTM only'
        print(f"  {name:30s}: {rmse:10.4f}{tag}")

# 逐县对比
print(f"\n{'='*70}")
print("逐县 RMSE 对比: LSTM vs LSTM+GNN")
print(f"{'='*70}")

lstm_wins, gnn_wins, ties = 0, 0, 0
diffs = []
for loc in locations:
    r_lstm = model_county_rmses.get(loc, float('inf'))
    r_gnn = gnn_county_rmses.get(loc, float('inf'))
    diff = r_lstm - r_gnn  # 正数 = GNN 更好
    diffs.append(diff)
    if diff > 0.1: gnn_wins += 1
    elif diff < -0.1: lstm_wins += 1
    else: ties += 1

print(f"  GNN 更好: {gnn_wins}/{L} 县")
print(f"  LSTM 更好: {lstm_wins}/{L} 县")
print(f"  持平: {ties}/{L} 县")
print(f"  RMSE 差值: mean={np.mean(diffs):.2f}, median={np.median(diffs):.2f}")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 散点图
ax = axes[0]
x_lstm = [model_county_rmses[loc] for loc in locations]
y_gnn = [gnn_county_rmses[loc] for loc in locations]
ax.scatter(x_lstm, y_gnn, alpha=0.6, s=40, c='#2196F3', edgecolors='white')
maxv = max(max(x_lstm), max(y_gnn))
ax.plot([0, maxv], [0, maxv], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('LSTM-only RMSE'); ax.set_ylabel('LSTM+GNN RMSE')
ax.set_title(f'各县 RMSE: LSTM vs LSTM+GNN\n(对角线下方 = GNN 更好, {gnn_wins}/{L} 县)')
ax.legend()

# RMSE 差值排序
ax = axes[1]
sorted_diffs = sorted(zip(locations, diffs), key=lambda x: x[1], reverse=True)
colors_d = ['#4CAF50' if d > 0 else '#F44336' for _, d in sorted_diffs]
ax.barh(range(L), [d for _, d in sorted_diffs], color=colors_d, alpha=0.7)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('RMSE 差值 (LSTM - GNN, 正=GNN更好)')
ax.set_title('各县: GNN 带来的 RMSE 变化')
ax.set_yticks([])

# 训练曲线对比
ax = axes[2]
ax.plot(history['val_rmse'], label='LSTM only', color='#FF9800', alpha=0.8)
ax.plot(history_gnn['val_rmse'], label='LSTM+GNN', color='#2196F3', alpha=0.8)
ax.axhline(best_val_rmse, color='#FF9800', ls='--', alpha=0.4)
ax.axhline(best_gnn_rmse, color='#2196F3', ls='--', alpha=0.4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val RMSE'); ax.set_title('训练曲线对比')
ax.legend()

plt.suptitle(f'Deep LSTM vs LSTM+GNN | GNN 提升 {gnn_wins}/{L} 县', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

if WANDB_ENABLED:
    wandb.log({'charts/lstm_vs_gnn': wandb.Image(fig)})
    for k, v in gnn_results.items():
        wandb.summary[f'eval_gnn/{k}'] = v

In [ ]:
# §P2.5: 最终总结

print(f"\n{'='*70}")
print("实验总结")
print(f"{'='*70}")
print(f"\nPhase 1: {MODEL_NAME}")
print(f"  Best Val RMSE: {best_val_rmse:.4f} @ epoch {best_epoch}")
print(f"  参数量: {n_params:,}")
for k,v in sorted(model_results.items(), key=lambda x: x[1]):
    print(f"  {k}: {v:.4f}")

print(f"\nPhase 2: {GNN_MODEL_NAME}")
print(f"  Best Val RMSE: {best_gnn_rmse:.4f} @ epoch {best_gnn_epoch}")
print(f"  参数量: {n_params_gnn:,}")
for k,v in sorted(gnn_results.items(), key=lambda x: x[1]):
    print(f"  {k}: {v:.4f}")

lstm_24 = model_results.get(f'{MODEL_NAME}_24h', float('inf'))
gnn_24 = gnn_results.get(f'{GNN_MODEL_NAME}_24h', float('inf'))
print(f"\nGNN 在 24h 上的提升: {lstm_24:.2f} → {gnn_24:.2f} ({(lstm_24-gnn_24)/lstm_24*100:+.1f}%)")

print(f"\n交付物:")
print(f"  {RESULTS_DIR}/{MODEL_NAME}_best.pt")
print(f"  {RESULTS_DIR}/{MODEL_NAME}_val_pred.npy")
print(f"  {RESULTS_DIR}/{GNN_MODEL_NAME}_best.pt")
print(f"  {RESULTS_DIR}/{GNN_MODEL_NAME}_val_pred.npy")

if WANDB_ENABLED:
    wandb.summary['phase1_best_rmse'] = best_val_rmse
    wandb.summary['phase2_best_rmse'] = best_gnn_rmse
    wandb.summary['gnn_improvement_24h'] = lstm_24 - gnn_24
    wandb.finish()
    print("\n✓ wandb finished")

---
## §8 提交文件生成

In [ ]:
# §8 提交文件
GENERATE_SUBMISSION = False  # TODO: 确认模型效果后改为 True

if GENERATE_SUBMISSION:
    # TODO: 加载 test data + 推理 + 生成 CSV
    # ds_test_24h = xr.open_dataset(os.path.join(DATA_DIR, 'test_24h_demo.nc'))
    # ds_test_48h = xr.open_dataset(os.path.join(DATA_DIR, 'test_48h_demo.nc'))
    # ...
    print('⚠ 请完成 test set 推理代码')
else:
    print('跳过 (GENERATE_SUBMISSION=False)')

In [ ]:
# §9 (被 §P2.5 替代)
print("见上方 §P2.5 总结")

---
## 附录

### 调参优先级
1. **Loss** — `huber` 或 `weighted_mse` (极端值友好)
2. **LR** — {1e-4, 3e-4, 1e-3, 3e-3}，Transformer 偏小
3. **Depth** — layers × hidden_dim
4. **SEQ_LEN** — {24, 48, 72}，Attention/Transformer 受益于长序列
5. **进阶模块** — GNN, Two-Stage, Ensemble

### Phase 1 关键发现
- 停电滞后特征贡献 60%+ → 确保 `OUTAGE_LAGS` 包含 24h
- 极端县 (26125, 26163) RMSE 极高 → 尝试县级 re-weighting
- 70% 零值 → Two-Stage (先分类再回归) 可能有效
- 天气 rolling 特征比 raw 更有用 → `weather_rolling` 组

### 分工交付协议
每人完成后提供:
1. `results/{MODEL_NAME}_best.pt` — checkpoint
2. `results/{MODEL_NAME}_val_pred.npy` — 验证集预测 `(T_val, L)`
3. wandb 上的完整实验记录
4. 24h / 48h RMSE 数字 → 汇总到排行榜